In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 0 MiB


In [7]:
!pip install transformers datasets evaluate numpy torch peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [8]:
!pip install rouge-score

In [9]:
from datasets import load_from_disk
from transformers import BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import BartForConditionalGeneration, BartConfig
import evaluate
import numpy as np
import torch
import os
import logging
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import gc

In [10]:
"""
CNN/DailyMail 数据集预处理脚本
用于 BART 模型适配策略对比研究：全参数微调、超轻量微调、LoRA 与从零训练

预处理步骤：
1. 加载 CNN/DailyMail 数据集
2. 最小化文本清洗（仅标准化空白字符）
3. 使用 BART 分词器进行分词（最大长度 512）
4. 划分训练集、验证集和测试集
5. 保存为 Hugging Face Dataset 格式
"""

from datasets import load_dataset, DatasetDict
from transformers import BartTokenizer, BartModel
import logging
from typing import Dict, List

#from google.colab import drive
#drive.mount('/content/drive')

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

_train_size = 20000
_val_size = 2000
_test_size = 2000

class CNNDailyMailPreprocessor:
    """CNN/DailyMail 数据集预处理类"""

    def __init__(self, model_name: str = "facebook/bart-base", max_length: int = 1024):
        """
        初始化预处理器

        Args:
            model_name: BART 模型名称
            max_length: 最大序列长度
        """
        self.model_name = model_name
        self.max_length = max_length

        # 加载 BART 分词器
        logger.info(f"加载分词器：{model_name}")
        self.tokenizer = BartTokenizer.from_pretrained(model_name)

        logger.info("预处理器初始化完成")

    def preprocess_text(self, text: str) -> str:
        """
        最小化文本预处理（仅标准化空白字符）

        Args:
            text: 原始文本

        Returns:
            处理后的文本
        """
        if not text:
            return ""

        # 仅标准化空白字符，保留原始大小写、标点和数字
        text = ' '.join(text.split()).strip()

        return text

    def tokenize_function(self, examples: Dict[str, List]) -> Dict[str, List]:
        """
        分词函数

        Args:
            examples: 包含文章和摘要的字典

        Returns:
            分词后的字典
        """
        # 最小化预处理文章
        articles = [self.preprocess_text(article) for article in examples['article']]

        # 最小化预处理摘要
        highlights = [self.preprocess_text(highlight) for highlight in examples['highlights']]

        # 对文章进行分词
        model_inputs = self.tokenizer(
            articles,
            max_length=self.max_length,
            truncation=True,
            add_special_tokens=True,
            padding="max_length"
        )

        # 对摘要进行分词（作为标签）
        labels = self.tokenizer(
            highlights,
            max_length=256,
            truncation=True,
            add_special_tokens=True,
            padding="max_length"
        )

        model_inputs["labels"] = labels["input_ids"]

        return model_inputs

    def load_and_preprocess(self) -> DatasetDict:
        """
        加载并预处理数据集

        Returns:
            预处理后的 DatasetDict
        """
        logger.info(f"加载数据集：cnn_dailymail (版本：3.0.0)")

        # 加载数据集
        dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

        # 子集大小（用于快速原型开发）
        train_size = min(_train_size, len(dataset['train']))
        val_size = min(_val_size, len(dataset['validation']))
        test_size = min(_test_size, len(dataset['test']))

        # 创建子集
        train_subset = dataset['train'].select(range(train_size))
        val_subset = dataset['validation'].select(range(val_size))
        test_subset = dataset['test'].select(range(test_size))

        logger.info(f"训练集大小：{len(train_subset)}")
        logger.info(f"验证集大小：{len(val_subset)}")
        logger.info(f"测试集大小：{len(test_subset)}")

        # 应用预处理
        logger.info("开始预处理数据集...")

        train_processed = train_subset.map(
            self.tokenize_function,
            batched=True,
            batch_size=8,
            remove_columns=['article', 'highlights', 'id']
        )

        val_processed = val_subset.map(
            self.tokenize_function,
            batched=True,
            batch_size=8,
            remove_columns=['article', 'highlights', 'id']
        )

        test_processed = test_subset.map(
            self.tokenize_function,
            batched=True,
            batch_size=8,
            remove_columns=['article', 'highlights', 'id']
        )

        # 创建 DatasetDict
        processed_datasets = DatasetDict({
            'train': train_processed,
            'validation': val_processed,
            'test': test_processed
        })

        logger.info("数据集预处理完成")

        return processed_datasets

    def save_dataset(self, dataset: DatasetDict, save_path: str = "./processed_cnn_dailymail"):
        """
        保存预处理后的数据集

        Args:
            dataset: 预处理后的数据集
            save_path: 保存路径
        """
        logger.info(f"保存数据集到：{save_path}")
        dataset.save_to_disk(save_path)
        logger.info("数据集保存完成")

    def get_dataset_info(self, dataset: DatasetDict):
        """显示数据集信息"""
        logger.info("数据集信息:")
        for split in dataset.keys():
            logger.info(f"  {split}: {len(dataset[split])} 个样本")

            if len(dataset[split]) > 0:
                sample = dataset[split][0]
                logger.info(f"    样本键：{list(sample.keys())}")

                # 原始 tokens (ID 列表)
                input_tokens = sample['input_ids']
                label_tokens = sample['labels']

                # 解码第一个样本查看
                input_text = self.tokenizer.decode(input_tokens, skip_special_tokens=True)
                label_text = self.tokenizer.decode(label_tokens, skip_special_tokens=True)

                logger.info(f"    输入原始 Tokens : {input_tokens[:]}")
                logger.info(f"    标签原始 Tokens : {label_tokens[:]}")

                logger.info(f"    输入文本 (前 100 字符): {input_text[:100]}...")
                logger.info(f"    标签文本 (前 100 字符): {label_text[:100]}...")

def process_data():
    """主函数"""
    logger.info("开始 CNN/DailyMail 数据集预处理")

    # 初始化预处理器
    preprocessor = CNNDailyMailPreprocessor(
        model_name="facebook/bart-base",
        max_length=1024
    )

    # 加载并预处理数据集
    processed_datasets = preprocessor.load_and_preprocess()

    # 显示数据集信息
    preprocessor.get_dataset_info(processed_datasets)

    # 保存数据集
    preprocessor.save_dataset(processed_datasets, "./processed_cnn_dailymail")

    logger.info("预处理流程完成")

process_data()

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

In [11]:
# ==================== Configuration ====================
MODEL_NAME = "facebook/bart-base"
MAX_TARGET_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 2
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
seed = 42

# ==================== LoRA Configuration ====================
# LoRA 超参数配置
LORA_R = 32  # LoRA 秩，通常为 4, 8, 16
LORA_ALPHA = 64  # LoRA 缩放参数
LORA_DROPOUT = 0.1  # LoRA dropout 率
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj"]  # BART 的目标模块

# 配置 LoRA
lora_config = LoraConfig(
    r=LORA_R,  # 秩
    lora_alpha=LORA_ALPHA,  # 缩放参数
    target_modules=LORA_TARGET_MODULES,  # 要应用 LoRA 的模块
    lora_dropout=LORA_DROPOUT,  # dropout
    bias="none",  # 是否训练偏置
    task_type=TaskType.SEQ_2_SEQ_LM,  # 任务类型：序列到序列语言模型
)

# ==================== Load Model & Tokenizer ====================
tokenizer = BartTokenizer.from_pretrained(MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

# ==================== Load Datasets ====================
# Load from single DatasetDict directory
processed_datasets = load_from_disk("./processed_cnn_dailymail")

# Extract splits
train_tokenized_data = processed_datasets['train']
train_tokenized_data = train_tokenized_data.shuffle(seed=seed).select(range(20000))

validation_tokenized_data = processed_datasets['validation']
validation_tokenized_data = validation_tokenized_data.shuffle(seed=seed).select(range(200))

test_tokenized_data = processed_datasets['test']
test_tokenized_data = test_tokenized_data.shuffle(seed=seed).select(range(200))

# ==================== Training Arguments ====================
def training_args(logging_name="test", LEARNING_RATE = 1e-4,NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=4,EVAL_BATCH_SIZE=4):
    return Seq2SeqTrainingArguments(
        output_dir=f"bart_cnndailymail_model/{logging_name}",
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        save_strategy="no",
        eval_strategy="no",
        save_total_limit=1,
        #load_best_model_at_end=True,
        metric_for_best_model="rougeL",
        greater_is_better=True,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        generation_num_beams=2,
        logging_dir=f"./logs/{logging_name}",
        logging_strategy="steps",
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to="tensorboard",
    )

# ==================== ROUGE Metric ====================
metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # 确保 tokenizer 有 pad_token_id
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id  # BART 中 eos_token_id 常用作 pad

    # Replace -100 with pad_token_id for decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)

    # Find indices of invalid prediction tokens
    invalid_pred_mask = (predictions < 0) | (predictions >= tokenizer.vocab_size)
    if np.any(invalid_pred_mask):
        invalid_indices = np.argwhere(invalid_pred_mask)
        invalid_values = predictions[invalid_pred_mask]
        print(f" INVALID PREDICTION TOKENS FOUND: {len(invalid_values)}")
        print(f" Sample invalid indices: {invalid_indices[:10]}")
        print(f" Sample invalid values: {invalid_values[:10]}")

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Normalize for ROUGE (optional but recommended)
    decoded_preds = ["\n".join(pred.strip().split("\n")) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split("\n")) for label in decoded_labels]

    # Compute ROUGE scores
    result = metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
        rouge_types=["rouge1", "rouge2", "rougeL"]
    )

    # Convert to percentage
    result = {key: round(value * 100, 4) for key, value in result.items()}

    # Add mean length for reference
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return result

# ==================== Bitfit model function====================
def apply_bitfit(model):
    """冻结所有参数，仅解冻 bias 和 LayerNorm 参数（BitFit 风格）"""
    # 1. 先冻结所有参数
    for param in model.parameters():
        param.requires_grad = False

    # 2. 解冻所有 bias 参数（包括 LayerNorm 的 bias）
    for name, param in model.named_parameters():
        if "bias" in name:                     # 包含 "bias" 的（全连接、LayerNorm 等）
            param.requires_grad = True
        elif "LayerNorm" in name:              # LayerNorm 的 weight 和 bias（可选）
            param.requires_grad = True
        # 如果只想要 bias，就只保留第一个条件；如果想要同时更新 LayerNorm 的权重，则加上第二个条件

    #打印可训练参数数量
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"BitFit: Trainable params: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")
    return model

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

# Bart-base： masked words prediction

In [31]:
from transformers import AutoTokenizer, BartForConditionalGeneration

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")


TXT = """
Manchester United <mask> Club,
commonly referred to as Man United (often stylised as Man Utd) or simply United,

"""
input_ids = tokenizer([TXT], return_tensors="pt")["input_ids"]
logits = model(input_ids).logits
masked_index = (input_ids[0] == tokenizer.mask_token_id).nonzero().item()
probs = logits[0, masked_index].softmax(dim=0)
values, predictions = probs.topk(5)

# Print formatted results
print("=" * 60)
print("DEMO 1: Mask Filling with BART-base Pre-trained Model")
print("=" * 60)

print(TXT)

print(f"{'Rank':<5} | {'Token':<20} | {'Probability':<10}")
print("-" * 40)
for i, (pred, val) in enumerate(zip(predictions, values)):
    token = tokenizer.decode(pred)
    prob = val.item() * 100
    print(f"{i+1:<5} | {token:<20} | {prob:.2f}%")



Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

DEMO 1: Mask Filling with BART-base Pre-trained Model

Manchester United <mask> Club, 
commonly referred to as Man United (often stylised as Man Utd) or simply United, 


Rank  | Token                | Probability
----------------------------------------
1     |  Football            | 68.31%
2     |  (                   | 5.17%
3     |  Supporters          | 3.75%
4     |  Club                | 2.06%
5     | ,                    | 1.95%


# # 使用配置创建随机初始化的模型（不加载预训练权重）

In [ ]:
'''epoch=2, base, 使用2e-4保持参数一致'''
# 获取原始模型的配置
config = BartConfig.from_pretrained(MODEL_NAME)
# 使用配置创建随机初始化的模型（不加载预训练权重）
base_model = BartForConditionalGeneration(config)
# Ensure pad_token_id is set correctly
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.decoder_start_token_id = tokenizer.bos_token_id

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

trainer_base = Seq2SeqTrainer(
    model=base_model,
    args=training_args(logging_name="base",LEARNING_RATE=2e-4,NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=16,EVAL_BATCH_SIZE=16),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics,
)

print("Starting training base model...")
trainer_base.train()
# 保存模型
trainer_base.save_model("bart_cnndailymail_model_base")
print(f"Base initialization from scratch completed.")

print("Starting evaling initialization from scratch model...")
eval_results = trainer_base.evaluate()
items = list(eval_results.items())

print("\n" + "="*100)
print("Initialization Evaluation Results".center(100))
print("="*100)

# 每四个数据显示一行
for i in range(0, len(items), 4):
    row_items = items[i:i+4]
    formatted_items = []
    for metric_name, metric_value in row_items:
        if isinstance(metric_value, float):
            formatted_items.append(f"{metric_name}: {metric_value:.4f}")
        else:
            formatted_items.append(f"{metric_name}: {metric_value}")
    print("  ".join(formatted_items))
print("="*100)

# Batch_size影响显存显示
max_allocated = torch.cuda.max_memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Training Memory: {max_allocated:.3f} GB / Total Memory: {total:.1f} GB")
del base_model
del trainer_base
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training base model...


Step,Training Loss
100,9.124365
200,1.438091
300,0.402134
400,0.378257
500,0.367496
600,0.357906
700,0.350174
800,0.339226
900,0.334662
1000,0.330456


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Base initialization from scratch completed.
Starting evaling initialization from scratch model...



                                 Initialization Evaluation Results                                  
eval_loss: 0.3048  eval_rouge1: 14.6154  eval_rouge2: 1.0255  eval_rougeL: 11.4194
eval_gen_len: 76.0000  eval_runtime: 16.2633  eval_samples_per_second: 12.2980  eval_steps_per_second: 0.7990
epoch: 2.0000
Training Memory: 19.415 GB / Total Memory: 39.5 GB


# 1. full-fine-tuning

In [20]:
'''epoch=2, Full-Fine-Tune, 全参数理论最优学习率3e-5, 使用2e-4保持参数一致'''
fft_model = BartForConditionalGeneration.from_pretrained(MODEL_NAME)
# Ensure pad_token_id is set correctly
fft_model.config.pad_token_id = tokenizer.pad_token_id
fft_model.config.decoder_start_token_id = tokenizer.bos_token_id

trainer_fft = Seq2SeqTrainer(
    model=fft_model,
    args=training_args(logging_name="full-ft",LEARNING_RATE=2e-4,NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=16,EVAL_BATCH_SIZE=16),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics,
)

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

print("Starting training Full-FT model...")
trainer_fft.train()
# 保存模型
trainer_fft.save_model("bart_cnndailymail_model_fft")
print(f"Full fine-tuning completed.")

print("Starting evaling Full-FT model...")
eval_results = trainer_fft.evaluate()
items = list(eval_results.items())

print("\n" + "="*100)
print("Full-FT Evaluation Results".center(100))
print("="*100)

# 每四个数据显示一行
for i in range(0, len(items), 4):
    row_items = items[i:i+4]
    formatted_items = []
    for metric_name, metric_value in row_items:
        if isinstance(metric_value, float):
            formatted_items.append(f"{metric_name}: {metric_value:.4f}")
        else:
            formatted_items.append(f"{metric_name}: {metric_value}")
    print("  ".join(formatted_items))
print("="*100)

#Batch_size影响显存显示
max_allocated = torch.cuda.max_memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Training Memory: {max_allocated:.3f} GB / Total Memory: {total:.1f} GB")
del fft_model
del trainer_fft
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training Full-FT model...


Step,Training Loss
100,5.771399
200,0.576290
300,0.567220
400,0.561361
500,0.539897
600,0.533958
700,0.525946
800,0.509652
900,0.505341
1000,0.499293


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full fine-tuning completed.
Starting evaling Full-FT model...



                                     Full-FT Evaluation Results                                     
eval_loss: 0.4332  eval_rouge1: 35.8901  eval_rouge2: 14.4698  eval_rougeL: 24.1254
eval_gen_len: 58.4050  eval_runtime: 21.5903  eval_samples_per_second: 9.2630  eval_steps_per_second: 0.6020
epoch: 2.0000
Training Memory: 9.846 GB / Total Memory: 39.5 GB


### full fine tuning demo

In [24]:
article = """
Abstract - This report presents a study of
Bilibili’s recommender system through six
technical publications, examining its
technological evolution, user-centric
challenges, and improvement pathways. Three
core technological pillars are identified: high-
performance inference architecture, unified
feature consistency frameworks, and advanced
feature interaction modeling. Grounded in
recommender system theory and personal user
experience, this report critically evaluates
c u r r e n t l i m i t a t i o n s a n d p r o p o s e s
improvements including unbiased interest
modeling and multimodal large language
models for enhanced content understanding.
Bridging industrial practice with academic
theory, this study offers a reference for next-
generation AI-driven recommender systems.
Keywords—Bilibili App, Recommender
Systems; Industrial Architecture; Data
Consistency; Feature Interaction; Multimodal
Learning.
I. INTRODUCTION
In the era of information overload,
recommender systems have become the central
nervous system of content platforms, shaping user
experience and platform ecology. Bilibili’s
recommender system serves a dual mission:
maximizing user engagement while preserving its
unique community identity. Serving hundreds of
millions of users across heterogeneous content
types demands a balance between accuracy,
robustness, low latency, and scalability.
The evolution of Bilibili’s architecture reflects
continuous responses to scaling challenges. As
documented in Sun Jiaqi’s 2022 blog, the system
transitioned from coarse-grained to fine-grained
retrieval, achieving ten-second granularity
danmaku indexing with the self-developed
Taishan KV database handling millions of QPS
[4].
However, rapid iteration has surfaced
persistent challenges. The Bilibili Tech Team’s
2024 blog highlights that data inconsistency
remains critical: discrepancies in feature
timestamps, delayed content exposure, and
implementation differences between training and
serving lead to “training-serving skew,” where
offline metrics fail to predict online performance
[1]. Such literature emphasizes engineering
solutions with limited attention to user-centered
perspectives.
This report adopts a dual perspective: it
synthesizes the literature to distill core
architectural innovations, then critically analyzes
the system through user perception and the “Five
Ws” dimensions, examining whether engineering-
centric optimizations adequately address user
needs. Integrating personal experience with
theory, this report proposes improvements such as
incorporating multimodal large language models
to enhance content understanding.
"""

import os
from transformers import AutoTokenizer, BartForConditionalGeneration, BartConfig
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

# Resolve absolute path to avoid HFValidationError
model_path = os.path.abspath("./bart_cnndailymail_model_fft")

# Check if the model directory exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Local model directory not found: {model_path}")

# Force bart-base config to match the weights (768 hidden size)
config = BartConfig.from_pretrained("facebook/bart-base")

# Load model with explicit config
ft_model = BartForConditionalGeneration.from_pretrained(
    model_path,
    config=config,
    local_files_only=True
)
print("=" * 60)
print("DEMO 2: News Summarization with BART-base Full Fine-Tuning Model")
print("=" * 60)

# Tokenize input
input_ids = tokenizer.encode(article, return_tensors="pt", max_length=1024)

# Generate summary
summary_ids = ft_model.generate(input_ids, max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)

# Decode and print result
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"### Original Article:\n{article[:500]}","..............")
print(f"\n### Generated Summary:\n{summary}")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

DEMO 2: News Summarization with BART-base Full Fine-Tuning Model
### Original Article:

Abstract - This report presents a study of
Bilibili’s recommender system through six
technical publications, examining its
technological evolution, user-centric
challenges, and improvement pathways. Three
core technological pillars are identified: high-
performance inference architecture, unified
feature consistency frameworks, and advanced
feature interaction modeling. Grounded in
recommender system theory and personal user
experience, this report critically evaluates
c u r r e n t l i m i t a ..............

### Generated Summary:
Bilibili's recommender system serves a dual mission:maximizing user engagement and preserving community identity . The evolution of Bilibili’s architecture reflects itscontinuous responses to scaling challenges .


# bitfit

In [26]:
'''epoch=2, Bitfit 理论最优学习率2e-4'''
# 打印可训练参数信息
print("BitFit Model trainable parameter information")
bitfit_model = apply_bitfit(BartForConditionalGeneration.from_pretrained(MODEL_NAME))
bitfit_model.config.pad_token_id = tokenizer.pad_token_id
bitfit_model.config.decoder_start_token_id = tokenizer.bos_token_id

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

trainer_bitfit = Seq2SeqTrainer(
    model=bitfit_model,
    args=training_args(logging_name="bitfit", LEARNING_RATE=2e-4, NUM_TRAIN_EPOCHS=2, TRAIN_BATCH_SIZE=16, EVAL_BATCH_SIZE=16),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics
)

print("Starting training BitFit model...")
trainer_bitfit.train()
trainer_bitfit.save_model("bart_cnndailymail_model_bitfit")

print("Starting evaling BitFit model...")
eval_results = trainer_bitfit.evaluate()
items = list(eval_results.items())

print("\n" + "="*100)
print("BitFit Evaluation Results".center(100))
print("="*100)

# 每四个数据显示一行
for i in range(0, len(items), 4):
    row_items = items[i:i+4]
    formatted_items = []
    for metric_name, metric_value in row_items:
        if isinstance(metric_value, float):
            formatted_items.append(f"{metric_name}: {metric_value:.4f}")
        else:
            formatted_items.append(f"{metric_name}: {metric_value}")
    print("  ".join(formatted_items))
print("="*100)

#Batch_size影响显存显示
max_allocated = torch.cuda.max_memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Training Memory: {max_allocated:.3f} GB / Total Memory: {total:.1f} GB")
del bitfit_model
del trainer_bitfit
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print("BitFit fine-tuning completed.")

BitFit Model trainable parameter information


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


BitFit: Trainable params: 125,952 / 139,420,416 (0.09%)
Starting training BitFit model...


Step,Training Loss
100,12.696235
200,9.450269
300,6.141986
400,5.275715
500,4.950150
600,4.776243
700,4.676399
800,4.593762
900,4.526424
1000,4.479844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Starting evaling BitFit model...



                                     BitFit Evaluation Results                                      
eval_loss: 3.8518  eval_rouge1: 33.1561  eval_rouge2: 13.4734  eval_rougeL: 22.9441
eval_gen_len: 58.5950  eval_runtime: 27.1426  eval_samples_per_second: 7.3680  eval_steps_per_second: 0.4790
epoch: 2.0000
Training Memory: 7.070 GB / Total Memory: 39.5 GB
BitFit fine-tuning completed.


# bitfit demo

In [29]:
import os
from transformers import AutoTokenizer, BartForConditionalGeneration, BartConfig
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

# Resolve absolute path to avoid HFValidationError
model_path = os.path.abspath("./bart_cnndailymail_model_bitfit")

# Check if the model directory exists
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Local model directory not found: {model_path}")

# Force bart-base config to match the weights (768 hidden size)
config = BartConfig.from_pretrained("facebook/bart-base")

# Load model with explicit config
ft_model = BartForConditionalGeneration.from_pretrained(
    model_path,
    config=config,
    local_files_only=True
)
print("=" * 60)
print("DEMO 3: News Summarization with BART-base bitfit")
print("=" * 60)

# Tokenize input
input_ids = tokenizer.encode(article, return_tensors="pt", max_length=1024)

# Generate summary
summary_ids = ft_model.generate(input_ids, max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)

# Decode and print result
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"### Original Article:\n{article[:]}","..............")
print(f"\n### Generated Summary:\n{summary}")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

DEMO 2: News Summarization with BART-base Full Fine-Tuning Model
### Original Article:

Abstract - This report presents a study of
Bilibili’s recommender system through six
technical publications, examining its
technological evolution, user-centric
challenges, and improvement pathways. Three
core technological pillars are identified: high-
performance inference architecture, unified
feature consistency frameworks, and advanced
feature interaction modeling. Grounded in
recommender system theory and personal user
experience, this report critically evaluates
c u r r e n t l i m i t a t i o n s a n d p r o p o s e s
improvements including unbiased interest
modeling and multimodal large language
models for enhanced content understanding.
Bridging industrial practice with academic
theory, this study offers a reference for next-
generation AI-driven recommender systems.
Keywords—Bilibili App, Recommender
Systems; Industrial Architecture; Data
Consistency; Feature Interaction; Multimodal
Learn

# peft LoRA

In [15]:
'''epoch=2, LoRA 理论最优学习率2e-4'''
# 打印可训练参数信息
print("LoRA R=16 Model trainable parameter information：")
# 应用 LoRA适配器到模型
lora_model = get_peft_model(BartForConditionalGeneration.from_pretrained(MODEL_NAME), lora_config)
lora_model.print_trainable_parameters()
lora_model.config.pad_token_id = tokenizer.pad_token_id
lora_model.config.decoder_start_token_id = tokenizer.bos_token_id

#LoRA
trainer_lora = Seq2SeqTrainer(
    model=lora_model,
    args=training_args(logging_name="lora",LEARNING_RATE=2e-4,NUM_TRAIN_EPOCHS=2,TRAIN_BATCH_SIZE=16,EVAL_BATCH_SIZE=16),
    train_dataset=train_tokenized_data,
    eval_dataset=validation_tokenized_data,
    compute_metrics=compute_metrics,
)

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

print("Starting training LoRA model...")
trainer_lora.train()
# 保存 LoRA 适配器权重（轻量级）
lora_model.save_pretrained("bart_cnndailymail_model_lora")
# tokenizer.save_pretrained("bart_cnndailymail_model_lora")

print(f"LoRA fine-tuning completed.")
print(f"LoRA adapter saved to: bart_cnndailymail_model_lora")
print(f"Trainable parameters saved (much smaller than full model)")

print("Starting evaling LoRA model...")
eval_results = trainer_lora.evaluate()
items = list(eval_results.items())

print("\n" + "="*100)
print("LoRA Evaluation Results".center(100))
print("="*100)

# 每四个数据显示一行
for i in range(0, len(items), 4):
    row_items = items[i:i+4]
    formatted_items = []
    for metric_name, metric_value in row_items:
        if isinstance(metric_value, float):
            formatted_items.append(f"{metric_name}: {metric_value:.4f}")
        else:
            formatted_items.append(f"{metric_name}: {metric_value}")
    print("  ".join(formatted_items))
print("="*100)

#Batch_size影响显存显示
max_allocated = torch.cuda.max_memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Training Memory: {max_allocated:.3f} GB / Total Memory: {total:.1f} GB")
del lora_model
del trainer_lora
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

LoRA R=16 Model trainable parameter information：


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


trainable params: 3,538,944 || all params: 142,959,360 || trainable%: 2.4755
Starting training LoRA model...


Step,Training Loss
100,11.747579
200,5.335065
300,4.837203
400,4.806974
500,4.764059
600,4.751420
700,4.754771
800,4.741718
900,4.739472
1000,4.733155


LoRA fine-tuning completed.
LoRA adapter saved to: bart_cnndailymail_model_lora
Trainable parameters saved (much smaller than full model)
Starting evaling LoRA model...



                                      LoRA Evaluation Results                                       
eval_loss: 4.2498  eval_rouge1: 35.8748  eval_rouge2: 15.3221  eval_rougeL: 24.9524
eval_gen_len: 54.0700  eval_runtime: 23.8157  eval_samples_per_second: 8.3980  eval_steps_per_second: 0.5460
epoch: 2.0000
Training Memory: 8.624 GB / Total Memory: 39.5 GB


# peft lora demo

In [27]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from transformers import AutoTokenizer, BartForConditionalGeneration
import os
import torch


print("=" * 60)
print("DEMO 4: News Summarization with BART-base + LoRA Model")
print("=" * 60)

LORA_R = 32  # LoRA 秩，通常为 4, 8, 16
LORA_ALPHA = 64  # LoRA 缩放参数
LORA_DROPOUT = 0.1  # LoRA dropout 率
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "out_proj"]

# 1. Load base BART model without forcing potentially mismatched config
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base",)
lora_model = get_peft_model(BartForConditionalGeneration.from_pretrained(MODEL_NAME), lora_config)

lora_model.config.pad_token_id = tokenizer.pad_token_id
lora_model.config.decoder_start_token_id = tokenizer.bos_token_id

# 假设你已经有一个 PeftModel 实例 lora_model
lora_model.load_adapter("./bart_cnndailymail_model_lora", adapter_name="default")
# 然后设置当前使用的适配器
lora_model.set_adapter("default")


# 3. Set to evaluation mode
lora_model = lora_model.eval()

# 4. Generate summary
input_ids = tokenizer.encode(article, return_tensors="pt", max_length=1024)
with torch.no_grad():
    summary_ids = lora_model.generate(
        input_ids=input_ids,
        max_length=256,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"### Original Article:\n{article[:]}...\n")
print(f"### Generated Summary:\n{summary}")


DEMO 3: News Summarization with BART-base + LoRA Model


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

### Original Article:

Abstract - This report presents a study of
Bilibili’s recommender system through six
technical publications, examining its
technological evolution, user-centric
challenges, and improvement pathways. Three
core technological pillars are identified: high-
performance inference architecture, unified
feature consistency frameworks, and advanced
feature interaction modeling. Grounded in
recommender system theory and personal user
experience, this report critically evaluates
c u r r e n t l i m i t a t i o n s a n d p r o p o s e s
improvements including unbiased interest
modeling and multimodal large language
models for enhanced content understanding.
Bridging industrial practice with academic
theory, this study offers a reference for next-
generation AI-driven recommender systems.
Keywords—Bilibili App, Recommender
Systems; Industrial Architecture; Data
Consistency; Feature Interaction; Multimodal
Learning.
I. INTRODUCTION
In the era of information overload,
recommen